# Freddie Mac Mortgage Risk Project
## Notebook 01: Data Ingestion and Extraction

This notebook sets up the project environment, connects Google Drive, validates folder paths, extracts Freddie Mac sample files for 2019–2023, and verifies that the raw origination and servicing files are available for downstream analysis.

## 1. Install required packages

In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn pyarrow openpyxl joblib

## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Define project folder paths

In [3]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/freddie_mac_risk_project")
RAW = BASE / "data" / "raw"
DOCS = RAW / "docs"
INTERIM = BASE / "data" / "interim"
PROCESSED = BASE / "data" / "processed"
NOTEBOOKS = BASE / "notebooks"
SRC = BASE / "src"
OUTPUTS = BASE / "outputs"
FIGS = OUTPUTS / "figures"
TABLES = OUTPUTS / "tables"
MODEL_DIR = OUTPUTS / "model"
REPORT_DIR = OUTPUTS / "report"

for p in [INTERIM, PROCESSED, NOTEBOOKS, SRC, FIGS, TABLES, MODEL_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Base path exists:", BASE.exists())
print("Raw path exists:", RAW.exists())
print("Docs path exists:", DOCS.exists())

Base path exists: True
Raw path exists: True
Docs path exists: True


## 4. Verify raw Freddie Mac files

In [20]:
print("Files in RAW folder:")
for f in sorted(RAW.iterdir()):
    print("-", f.name)

print("\nFiles in DOCS folder:")
for f in sorted(DOCS.iterdir()):
    print("-", f.name)

Files in RAW folder:
- docs
- sample_2019.zip
- sample_2020.zip
- sample_2021.zip
- sample_2022.zip
- sample_2023.zip

Files in DOCS folder:
- user_guide.pdf


## 5. Validate expected sample zip files

In [21]:
expected_files = {
    "sample_2019.zip",
    "sample_2020.zip",
    "sample_2021.zip",
    "sample_2022.zip",
    "sample_2023.zip",
}

raw_files = {f.name for f in RAW.iterdir() if f.is_file()}
missing = expected_files - raw_files
extra = raw_files - expected_files

print("Missing files:", missing if missing else "None")
print("Extra files:", extra if extra else "None")

Missing files: None
Extra files: None


## 6. Extract Freddie Mac sample files

In [22]:
import zipfile

zip_files = [
    RAW / "sample_2019.zip",
    RAW / "sample_2020.zip",
    RAW / "sample_2021.zip",
    RAW / "sample_2022.zip",
    RAW / "sample_2023.zip",
]

for z in zip_files:
    out_dir = INTERIM / z.stem
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(z, "r") as zip_ref:
        zip_ref.extractall(out_dir)

print("All sample zip files extracted successfully.")

All sample zip files extracted successfully.


## 7. Inspect extracted files by vintage year

In [23]:
for year in [2019, 2020, 2021, 2022, 2023]:
    folder = INTERIM / f"sample_{year}"
    print(f"\nContents of {folder.name}:")
    for f in sorted(folder.iterdir()):
        print("-", f.name)


Contents of sample_2019:
- sample_orig_2019.txt
- sample_svcg_2019.txt

Contents of sample_2020:
- sample_orig_2020.txt
- sample_svcg_2020.txt

Contents of sample_2021:
- sample_orig_2021.txt
- sample_svcg_2021.txt

Contents of sample_2022:
- sample_orig_2022.txt
- sample_svcg_2022.txt

Contents of sample_2023:
- sample_orig_2023.txt
- sample_svcg_2023.txt


## 8. Validate required origination and servicing files

In [24]:
required_patterns = {
    "orig": "sample_orig_{}.txt",
    "svcg": "sample_svcg_{}.txt"
}

for year in [2019, 2020, 2021, 2022, 2023]:
    folder = INTERIM / f"sample_{year}"
    expected_orig = required_patterns["orig"].format(year)
    expected_svcg = required_patterns["svcg"].format(year)

    files_present = {f.name for f in folder.iterdir() if f.is_file()}

    print(f"\nYear {year}")
    print("Origination file present:", expected_orig in files_present)
    print("Servicing file present:", expected_svcg in files_present)


Year 2019
Origination file present: True
Servicing file present: True

Year 2020
Origination file present: True
Servicing file present: True

Year 2021
Origination file present: True
Servicing file present: True

Year 2022
Origination file present: True
Servicing file present: True

Year 2023
Origination file present: True
Servicing file present: True


## 9. Preview raw origination file format

In [25]:
sample_file = INTERIM / "sample_2019" / "sample_orig_2019.txt"

with open(sample_file, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(5):
        print(f.readline()[:300])

741|201903|N|204902||000|1|P|80|33|372000|80|5.125|R|N|FRM|WY|SF|82400|F19Q10000056|P|360|02|Other sellers|Other servicers|||9||7|N|7

731|201903|N|204902|13460|000|1|P|77|44|250000|77|4.875|R|N|FRM|OR|SF|97700|F19Q10000060|P|360|01|Other sellers|Other servicers|||9||7|N|7

722|201903|N|204902||30|1|P|95|41|261000|95|5.5|R|N|FRM|IN|SF|46700|F19Q10000084|P|360|02|Other sellers|Other servicers|||9||7|N|N

729|201903|N|204902||25|1|P|87|17|61000|87|5.09|R|N|FRM|MI|SF|49800|F19Q10000106|P|360|01|Other sellers|Other servicers|||9||7|N|N

773|201903|N|204902|33700|000|1|P|29|43|390000|29|5.375|R|N|FRM|CA|SF|95300|F19Q10000189|C|360|02|Other sellers|Other servicers|||9||7|N|7



## 10. Preview raw servicing file format

In [26]:
sample_file = INTERIM / "sample_2019" / "sample_svcg_2019.txt"

with open(sample_file, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(5):
        print(f.readline()[:300])

F19Q10000056|201902|372000.00|0|000|360|||||5.125|0.00||||||||||||||999||||||372000.00

F19Q10000056|201903|372000.00|0|001|359|||||5.125|0.00||||||||||||||999||||||372000.00

F19Q10000056|201904|371000.00|0|002|358|||||5.125|0.00||||||||||||||999||||||371000.00

F19Q10000056|201905|370000.00|0|003|357|||||5.125|0.00||||||||||||||999||||||370000.00

F19Q10000056|201906|370000.00|0|004|356|||||5.125|0.00||||||||||||||75||||||370000.00



## 11. Final extraction inventory

In [27]:
all_extracted_files = list(INTERIM.rglob("*"))
all_extracted_files = [f for f in all_extracted_files if f.is_file()]

print("Total extracted files:", len(all_extracted_files))
for f in all_extracted_files:
    print("-", f.relative_to(INTERIM))

Total extracted files: 10
- sample_2019/sample_orig_2019.txt
- sample_2019/sample_svcg_2019.txt
- sample_2020/sample_orig_2020.txt
- sample_2020/sample_svcg_2020.txt
- sample_2021/sample_orig_2021.txt
- sample_2021/sample_svcg_2021.txt
- sample_2022/sample_orig_2022.txt
- sample_2022/sample_svcg_2022.txt
- sample_2023/sample_orig_2023.txt
- sample_2023/sample_svcg_2023.txt


## Conclusion

The Freddie Mac sample files for 2019–2023 have been successfully extracted and validated. The next notebook will define the column structure and safely load the origination and servicing files into pandas for analysis.

## 12. Define Freddie Mac column names

In [4]:
ORIG_COLS = [
    "credit_score",
    "first_payment_date",
    "first_time_homebuyer_flag",
    "maturity_date",
    "msa",
    "mi_pct",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "channel",
    "prepayment_penalty_flag",
    "amortization_type",
    "property_state",
    "property_type",
    "postal_code",
    "loan_sequence_number",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "servicer_name",
    "super_conforming_flag",
    "pre_harp_loan_sequence_number",
    "program_indicator",
    "relief_refinance_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator"
]

PERF_COLS = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "repurchase_flag",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_deferred_upb",
    "due_date_last_paid_installment",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "misc_expenses",
    "actual_loss_calculation",
    "modification_cost",
    "step_modification_flag",
    "payment_deferral_flag",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost"
]

print("Origination columns:", len(ORIG_COLS))
print("Servicing columns:", len(PERF_COLS))

Origination columns: 32
Servicing columns: 31


## 13. Define sample file paths for one test year

In [29]:
orig_2019_path = INTERIM / "sample_2019" / "sample_orig_2019.txt"
svcg_2019_path = INTERIM / "sample_2019" / "sample_svcg_2019.txt"

print("Origination path exists:", orig_2019_path.exists())
print("Servicing path exists:", svcg_2019_path.exists())

print(orig_2019_path)
print(svcg_2019_path)

Origination path exists: True
Servicing path exists: True
/content/drive/MyDrive/freddie_mac_risk_project/data/interim/sample_2019/sample_orig_2019.txt
/content/drive/MyDrive/freddie_mac_risk_project/data/interim/sample_2019/sample_svcg_2019.txt


## 14. Load 2019 origination file into pandas


In [30]:
import pandas as pd

orig_2019 = pd.read_csv(
    orig_2019_path,
    sep="|",
    names=ORIG_COLS,
    header=None,
    dtype=str,
    low_memory=False
)

print("Origination shape:", orig_2019.shape)
orig_2019.head()

Origination shape: (50000, 32)


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_pct,num_units,occupancy_status,cltv,dti,...,num_borrowers,seller_name,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,relief_refinance_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator
0,741,201903,N,204902,NaN,000,1,P,80,33,...,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7
1,731,201903,N,204902,13460,000,1,P,77,44,...,01,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7
2,722,201903,N,204902,NaN,30,1,P,95,41,...,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,N
3,729,201903,N,204902,NaN,25,1,P,87,17,...,01,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,N
4,773,201903,N,204902,33700,000,1,P,29,43,...,02,Other sellers,Other servicers,NaN,NaN,9,NaN,7,N,7


## 15. Load 2019 servicing file into pandas

In [31]:
svcg_2019 = pd.read_csv(
    svcg_2019_path,
    sep="|",
    names=PERF_COLS,
    header=None,
    dtype=str,
    low_memory=False
)

print("Servicing shape:", svcg_2019.shape)
svcg_2019.head()

Servicing shape: (1847477, 31)


,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,repurchase_flag,modification_flag,zero_balance_code,zero_balance_effective_date,...,actual_loss_calculation,modification_cost,step_modification_flag,payment_deferral_flag,estimated_ltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_status_code,current_month_modification_cost
F19Q10000056,201902,372000.00,0,000,360,NaN,NaN,NaN,NaN,5.125,...,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN,372000.00
F19Q10000056,201903,372000.00,0,001,359,NaN,NaN,NaN,NaN,5.125,...,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN,372000.00
F19Q10000056,201904,371000.00,0,002,358,NaN,NaN,NaN,NaN,5.125,...,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN,371000.00
F19Q10000056,201905,370000.00,0,003,357,NaN,NaN,NaN,NaN,5.125,...,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN,370000.00
F19Q10000056,201906,370000.00,0,004,356,NaN,NaN,NaN,NaN,5.125,...,NaN,NaN,NaN,75,NaN,NaN,NaN,NaN,NaN,370000.00


## 16. Validate parsed structure

In [32]:
print("Origination columns loaded:", len(orig_2019.columns))
print("Expected origination columns:", len(ORIG_COLS))

print("Servicing columns loaded:", len(svcg_2019.columns))
print("Expected servicing columns:", len(PERF_COLS))

Origination columns loaded: 32
Expected origination columns: 32
Servicing columns loaded: 31
Expected servicing columns: 31


In [33]:
print("Origination sample row:")
display(orig_2019.iloc[[0]].T)

print("Servicing sample row:")
display(svcg_2019.iloc[[0]].T)

Origination sample row:


,0
credit_score,741
first_payment_date,201903
first_time_homebuyer_flag,N
maturity_date,204902
msa,NaN
mi_pct,000
num_units,1
occupancy_status,P
cltv,80
dti,33


Servicing sample row:


,F19Q10000056
loan_sequence_number,201902
monthly_reporting_period,372000.00
current_actual_upb,0
current_loan_delinquency_status,000
loan_age,360
remaining_months_to_legal_maturity,NaN
repurchase_flag,NaN
modification_flag,NaN
zero_balance_code,NaN
zero_balance_effective_date,5.125


## 17. Inspect basic structure and missingness

In [34]:
print("Origination dtypes:")
print(orig_2019.dtypes.head(10))

print("\nServicing dtypes:")
print(svcg_2019.dtypes.head(10))

Origination dtypes:
credit_score                 object
first_payment_date           object
first_time_homebuyer_flag    object
maturity_date                object
msa                          object
mi_pct                       object
num_units                    object
occupancy_status             object
cltv                         object
dti                          object
dtype: object

Servicing dtypes:
loan_sequence_number                  object
monthly_reporting_period              object
current_actual_upb                    object
current_loan_delinquency_status       object
loan_age                              object
remaining_months_to_legal_maturity    object
repurchase_flag                       object
modification_flag                     object
zero_balance_code                     object
zero_balance_effective_date           object
dtype: object


In [35]:
orig_missing = orig_2019.isna().mean().sort_values(ascending=False).head(10)
svcg_missing = svcg_2019.isna().mean().sort_values(ascending=False).head(10)

print("Top 10 origination missingness:")
print(orig_missing)

print("\nTop 10 servicing missingness:")
print(svcg_missing)

Top 10 origination missingness:
pre_harp_loan_sequence_number    0.99940
relief_refinance_indicator       0.99940
super_conforming_flag            0.96666
msa                              0.09148
maturity_date                    0.00000
credit_score                     0.00000
first_time_homebuyer_flag        0.00000
first_payment_date               0.00000
cltv                             0.00000
dti                              0.00000
dtype: float64

Top 10 servicing missingness:
due_date_last_paid_installment        0.999982
net_sales_proceeds                    0.999982
mi_recoveries                         0.999982
maintenance_and_preservation_costs    0.999982
legal_costs                           0.999982
expenses                              0.999982
non_mi_recoveries                     0.999982
misc_expenses                         0.999982
zero_balance_removal_upb              0.999982
taxes_and_insurance                   0.999982
dtype: float64


## 18. Save lightweight ingestion checks

In [36]:
ingestion_summary = pd.DataFrame({
    "dataset": ["orig_2019", "svcg_2019"],
    "rows": [orig_2019.shape[0], svcg_2019.shape[0]],
    "cols": [orig_2019.shape[1], svcg_2019.shape[1]]
})

ingestion_summary.to_csv(TABLES / "ingestion_summary_2019.csv", index=False)
ingestion_summary

,dataset,rows,cols
0,orig_2019,50000,32
1,svcg_2019,1847477,31


## Conclusion

The 2019 Freddie Mac origination and servicing files were successfully ingested into pandas using the defined schema. The next step will be to parse dates and numeric fields, then scale ingestion from one vintage year to all years from 2019 to 2023.

## 20. Define key date and numeric columns


In [5]:
orig_date_cols = [
    "first_payment_date",
    "maturity_date"
]

orig_numeric_cols = [
    "credit_score",
    "mi_pct",
    "num_units",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "original_loan_term",
    "num_borrowers"
]

svcg_date_cols = [
    "monthly_reporting_period",
    "zero_balance_effective_date",
    "due_date_last_paid_installment"
]

svcg_numeric_cols = [
    "current_actual_upb",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_interest_rate",
    "current_deferred_upb",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "misc_expenses",
    "actual_loss_calculation",
    "modification_cost",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "current_month_modification_cost"
]

print("Origination date cols:", orig_date_cols)
print("Origination numeric cols:", orig_numeric_cols)
print("Servicing date cols:", svcg_date_cols)
print("Servicing numeric cols:", svcg_numeric_cols)

Origination date cols: ['first_payment_date', 'maturity_date']
Origination numeric cols: ['credit_score', 'mi_pct', 'num_units', 'cltv', 'dti', 'original_upb', 'ltv', 'interest_rate', 'original_loan_term', 'num_borrowers']
Servicing date cols: ['monthly_reporting_period', 'zero_balance_effective_date', 'due_date_last_paid_installment']
Servicing numeric cols: ['current_actual_upb', 'loan_age', 'remaining_months_to_legal_maturity', 'current_interest_rate', 'current_deferred_upb', 'mi_recoveries', 'net_sales_proceeds', 'non_mi_recoveries', 'expenses', 'legal_costs', 'maintenance_and_preservation_costs', 'taxes_and_insurance', 'misc_expenses', 'actual_loss_calculation', 'modification_cost', 'estimated_ltv', 'zero_balance_removal_upb', 'delinquent_accrued_interest', 'current_month_modification_cost']


## 21. Create typed working copies


In [38]:
orig_2019_typed = orig_2019.copy()
svcg_2019_typed = svcg_2019.copy()

## 22. Parse origination date fields


In [39]:
for col in orig_date_cols:
    orig_2019_typed[col] = pd.to_datetime(
        orig_2019_typed[col],
        format="%Y%m",
        errors="coerce"
    )

orig_2019_typed[orig_date_cols].head()

,first_payment_date,maturity_date
0,2019-03-01,2049-02-01
1,2019-03-01,2049-02-01
2,2019-03-01,2049-02-01
3,2019-03-01,2049-02-01
4,2019-03-01,2049-02-01


## 23. Parse servicing date fields

In [40]:
for col in svcg_date_cols:
    svcg_2019_typed[col] = pd.to_datetime(
        svcg_2019_typed[col],
        format="%Y%m",
        errors="coerce"
    )

svcg_2019_typed[svcg_date_cols].head()

,monthly_reporting_period,zero_balance_effective_date,due_date_last_paid_installment
F19Q10000056,NaT,NaT,NaT
F19Q10000056,NaT,NaT,NaT
F19Q10000056,NaT,NaT,NaT
F19Q10000056,NaT,NaT,NaT
F19Q10000056,NaT,NaT,NaT


## 24. Parse origination numeric fields

In [41]:
for col in orig_numeric_cols:
    orig_2019_typed[col] = pd.to_numeric(orig_2019_typed[col], errors="coerce")

orig_2019_typed[orig_numeric_cols].dtypes

,0
credit_score,int64
mi_pct,int64
num_units,int64
cltv,int64
dti,int64
original_upb,int64
ltv,int64
interest_rate,float64
original_loan_term,int64
num_borrowers,int64


## 25. Parse servicing numeric fields

In [42]:
for col in svcg_numeric_cols:
    svcg_2019_typed[col] = pd.to_numeric(svcg_2019_typed[col], errors="coerce")

svcg_2019_typed[svcg_numeric_cols].dtypes

,0
current_actual_upb,float64
loan_age,int64
remaining_months_to_legal_maturity,float64
current_interest_rate,float64
current_deferred_upb,float64
mi_recoveries,float64
net_sales_proceeds,float64
non_mi_recoveries,float64
expenses,float64
legal_costs,float64


## 26. Validate parsed sample rows

In [43]:
print("Typed origination sample:")
display(orig_2019_typed.loc[:, [
    "loan_sequence_number",
    "first_payment_date",
    "maturity_date",
    "credit_score",
    "cltv",
    "dti",
    "original_upb",
    "interest_rate"
]].head())

print("Typed servicing sample:")
display(svcg_2019_typed.loc[:, [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "current_interest_rate"
]].head())

Typed origination sample:


,loan_sequence_number,first_payment_date,maturity_date,credit_score,cltv,dti,original_upb,interest_rate
0,F19Q10000056,2019-03-01,2049-02-01,741,80,33,372000,5.125
1,F19Q10000060,2019-03-01,2049-02-01,731,77,44,250000,4.875
2,F19Q10000084,2019-03-01,2049-02-01,722,95,41,261000,5.500
3,F19Q10000106,2019-03-01,2049-02-01,729,87,17,61000,5.090
4,F19Q10000189,2019-03-01,2049-02-01,773,29,43,390000,5.375


Typed servicing sample:


,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,current_interest_rate
F19Q10000056,201902,NaT,0.0,000,360,0.0
F19Q10000056,201903,NaT,0.0,001,359,0.0
F19Q10000056,201904,NaT,0.0,002,358,0.0
F19Q10000056,201905,NaT,0.0,003,357,0.0
F19Q10000056,201906,NaT,0.0,004,356,0.0


## 27. Check date parsing success

In [44]:
date_parse_summary = pd.DataFrame({
    "column": orig_date_cols + svcg_date_cols,
    "non_null_after_parse": [
        orig_2019_typed[c].notna().sum() for c in orig_date_cols
    ] + [
        svcg_2019_typed[c].notna().sum() for c in svcg_date_cols
    ],
    "total_rows_reference": [
        len(orig_2019_typed) for _ in orig_date_cols
    ] + [
        len(svcg_2019_typed) for _ in svcg_date_cols
    ]
})

date_parse_summary["parse_rate"] = (
    date_parse_summary["non_null_after_parse"] /
    date_parse_summary["total_rows_reference"]
)

date_parse_summary

,column,non_null_after_parse,total_rows_reference,parse_rate
0,first_payment_date,50000,50000,1.0
1,maturity_date,50000,50000,1.0
2,monthly_reporting_period,0,1847477,0.0
3,zero_balance_effective_date,0,1847477,0.0
4,due_date_last_paid_installment,0,1847477,0.0


## 28. Check numeric parsing success


In [45]:
numeric_check_cols = [
    "credit_score", "cltv", "dti", "original_upb", "interest_rate",
    "current_actual_upb", "loan_age", "current_interest_rate"
]

numeric_parse_summary = pd.DataFrame({
    "column": numeric_check_cols,
    "non_null_after_parse": [
        orig_2019_typed[c].notna().sum() if c in orig_2019_typed.columns
        else svcg_2019_typed[c].notna().sum()
        for c in numeric_check_cols
    ],
    "dataset": [
        "origination" if c in orig_2019_typed.columns else "servicing"
        for c in numeric_check_cols
    ]
})

numeric_parse_summary

,column,non_null_after_parse,dataset
0,credit_score,50000,origination
1,cltv,50000,origination
2,dti,50000,origination
3,original_upb,50000,origination
4,interest_rate,50000,origination
5,current_actual_upb,1847356,servicing
6,loan_age,1847477,servicing
7,current_interest_rate,1847477,servicing


## 29. Inspect delinquency status values


In [46]:
delq_value_counts = (
    svcg_2019_typed["current_loan_delinquency_status"]
    .value_counts(dropna=False)
    .head(20)
)

delq_value_counts

,count
current_loan_delinquency_status,
003,49928
002,49839
004,49816
005,49560
006,49016
001,48943
007,48090
008,47002
000,45963


## 30. Save typed validation summaries

In [47]:
date_parse_summary.to_csv(TABLES / "date_parse_summary_2019.csv", index=False)
numeric_parse_summary.to_csv(TABLES / "numeric_parse_summary_2019.csv", index=False)
delq_value_counts.to_csv(TABLES / "delq_value_counts_2019.csv")

## Conclusion

The 2019 Freddie Mac origination and servicing files were successfully converted from raw text into typed working datasets. The next step will scale this ingestion and typing logic to all vintages from 2019 to 2023 before building the 12-month serious delinquency target.

## Debugging note: servicing file schema validation

At this stage, the origination data has parsed correctly, but the servicing data shows signs of column misalignment. In particular, key servicing fields such as `monthly_reporting_period` did not parse successfully, and the typed servicing preview suggests that values may have shifted across columns during import. Because the 12-month serious delinquency target will be built from the servicing file, it is critical to resolve this issue before scaling the workflow to all vintages. The next steps therefore focus on validating the raw servicing file structure, comparing the actual number of pipe-delimited fields to the defined schema, and reloading the file in a controlled way to identify the source of misalignment.

## 32. Diagnose servicing file structure


In [48]:
raw_svcg_path = INTERIM / "sample_2019" / "sample_svcg_2019.txt"

with open(raw_svcg_path, "r", encoding="utf-8", errors="ignore") as f:
    first_5_lines = [f.readline().strip() for _ in range(5)]

for i, line in enumerate(first_5_lines, start=1):
    split_fields = line.split("|")
    print(f"Line {i}: field count = {len(split_fields)}")
    print(split_fields[:10])   # first 10 raw fields only
    print("---")

Line 1: field count = 32
['F19Q10000056', '201902', '372000.00', '0', '000', '360', '', '', '', '']
---
Line 2: field count = 32
['F19Q10000056', '201903', '372000.00', '0', '001', '359', '', '', '', '']
---
Line 3: field count = 32
['F19Q10000056', '201904', '371000.00', '0', '002', '358', '', '', '', '']
---
Line 4: field count = 32
['F19Q10000056', '201905', '370000.00', '0', '003', '357', '', '', '', '']
---
Line 5: field count = 32
['F19Q10000056', '201906', '370000.00', '0', '004', '356', '', '', '', '']
---


## 33. Compare raw field count with servicing schema

In [49]:
raw_field_count = len(first_5_lines[0].split("|"))
defined_field_count = len(PERF_COLS)

print("Raw servicing field count:", raw_field_count)
print("Defined servicing column count:", defined_field_count)
print("Difference:", raw_field_count - defined_field_count)

Raw servicing field count: 32
Defined servicing column count: 31
Difference: 1


## 34. Reload servicing file with explicit index handling

In [50]:
svcg_2019_check = pd.read_csv(
    svcg_2019_path,
    sep="|",
    names=PERF_COLS,
    header=None,
    dtype=str,
    low_memory=False,
    index_col=False
)

print("Shape:", svcg_2019_check.shape)
display(svcg_2019_check.head())

/tmp/ipykernel_169/4041919523.py:1: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  svcg_2019_check = pd.read_csv(


Shape: (1847477, 31)


,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,repurchase_flag,modification_flag,zero_balance_code,zero_balance_effective_date,...,actual_loss_calculation,modification_cost,step_modification_flag,payment_deferral_flag,estimated_ltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_status_code,current_month_modification_cost
0,F19Q10000056,201902,372000.00,0,000,360,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
1,F19Q10000056,201903,372000.00,0,001,359,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
2,F19Q10000056,201904,371000.00,0,002,358,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
3,F19Q10000056,201905,370000.00,0,003,357,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
4,F19Q10000056,201906,370000.00,0,004,356,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,NaN,NaN


## 35. Correct the servicing schema

This section corrects the monthly servicing schema based on the Freddie Mac performance file layout. The earlier schema used an incorrect field name (`repurchase_flag`) and the servicing file also needs to be loaded with explicit index handling so pandas does not infer the first field as an index.

In [6]:
PERF_COLS = [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "defect_settlement_date",
    "modification_flag",
    "zero_balance_code",
    "zero_balance_effective_date",
    "current_interest_rate",
    "current_deferred_upb",
    "due_date_last_paid_installment",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "misc_expenses",
    "actual_loss_calculation",
    "modification_cost",
    "step_modification_flag",
    "payment_deferral_flag",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "delinquency_due_to_disaster",
    "borrower_assistance_status_code",
    "current_month_modification_cost"
]

print("Corrected servicing column count:", len(PERF_COLS))

Corrected servicing column count: 31


## 36. Reload 2019 servicing with corrected schema

In [52]:
svcg_2019 = pd.read_csv(
    svcg_2019_path,
    sep="|",
    names=PERF_COLS,
    header=None,
    dtype=str,
    low_memory=False,
    index_col=False
)

print("Reloaded servicing shape:", svcg_2019.shape)
display(svcg_2019.head())

/tmp/ipykernel_169/1707506716.py:1: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  svcg_2019 = pd.read_csv(


Reloaded servicing shape: (1847477, 31)


,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,defect_settlement_date,modification_flag,zero_balance_code,zero_balance_effective_date,...,actual_loss_calculation,modification_cost,step_modification_flag,payment_deferral_flag,estimated_ltv,zero_balance_removal_upb,delinquent_accrued_interest,delinquency_due_to_disaster,borrower_assistance_status_code,current_month_modification_cost
0,F19Q10000056,201902,372000.00,0,000,360,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
1,F19Q10000056,201903,372000.00,0,001,359,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
2,F19Q10000056,201904,371000.00,0,002,358,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
3,F19Q10000056,201905,370000.00,0,003,357,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,999,NaN,NaN,NaN,NaN,NaN
4,F19Q10000056,201906,370000.00,0,004,356,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,NaN,NaN


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 37. Recreate typed servicing dataset

In [53]:
svcg_2019_typed = svcg_2019.copy()

svcg_date_cols = [
    "monthly_reporting_period",
    "zero_balance_effective_date",
    "due_date_last_paid_installment",
    "defect_settlement_date"
]

svcg_numeric_cols = [
    "current_actual_upb",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_interest_rate",
    "current_deferred_upb",
    "mi_recoveries",
    "net_sales_proceeds",
    "non_mi_recoveries",
    "expenses",
    "legal_costs",
    "maintenance_and_preservation_costs",
    "taxes_and_insurance",
    "misc_expenses",
    "actual_loss_calculation",
    "modification_cost",
    "estimated_ltv",
    "zero_balance_removal_upb",
    "delinquent_accrued_interest",
    "current_month_modification_cost"
]

for col in svcg_date_cols:
    svcg_2019_typed[col] = pd.to_datetime(
        svcg_2019_typed[col],
        format="%Y%m",
        errors="coerce"
    )

for col in svcg_numeric_cols:
    svcg_2019_typed[col] = pd.to_numeric(
        svcg_2019_typed[col],
        errors="coerce"
    )

display(svcg_2019_typed.loc[:, [
    "loan_sequence_number",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_interest_rate"
]].head())

,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,current_interest_rate
0,F19Q10000056,2019-02-01,372000.0,0,0,360,5.125
1,F19Q10000056,2019-03-01,372000.0,0,1,359,5.125
2,F19Q10000056,2019-04-01,371000.0,0,2,358,5.125
3,F19Q10000056,2019-05-01,370000.0,0,3,357,5.125
4,F19Q10000056,2019-06-01,370000.0,0,4,356,5.125


## 38. Recheck servicing date parsing

In [54]:
svcg_date_parse_summary = pd.DataFrame({
    "column": svcg_date_cols,
    "non_null_after_parse": [svcg_2019_typed[c].notna().sum() for c in svcg_date_cols],
    "total_rows_reference": [len(svcg_2019_typed) for _ in svcg_date_cols]
})

svcg_date_parse_summary["parse_rate"] = (
    svcg_date_parse_summary["non_null_after_parse"] /
    svcg_date_parse_summary["total_rows_reference"]
)

svcg_date_parse_summary

,column,non_null_after_parse,total_rows_reference,parse_rate
0,monthly_reporting_period,1847477,1847477,1.000000
1,zero_balance_effective_date,35262,1847477,0.019087
2,due_date_last_paid_installment,35226,1847477,0.019067
3,defect_settlement_date,127,1847477,0.000069


## 39. Recheck delinquency value distribution

In [55]:
svcg_2019_typed["current_loan_delinquency_status"].value_counts(dropna=False).head(20)

,count
current_loan_delinquency_status,
0,1799008
1,17290
2,6338
3,4190
4,3057
5,2561
6,2079
7,1716
8,1530


## Note: scaling the validated ingestion workflow

The 2019 origination and servicing files have been successfully ingested, typed, and schema-validated. The next step is to apply the same logic across all sample vintages from 2019 to 2023, combine them into unified origination and servicing datasets, and save the typed outputs for downstream target creation and modeling.

## 40. Define analysis vintages

In [7]:
YEARS = [2019, 2020, 2021, 2022, 2023]
YEARS

[2019, 2020, 2021, 2022, 2023]

## 41. Create reusable data-loading functions

In [8]:
def get_orig_path(year):
    return INTERIM / f"sample_{year}" / f"sample_orig_{year}.txt"

def get_svcg_path(year):
    return INTERIM / f"sample_{year}" / f"sample_svcg_{year}.txt"

def load_orig_year(year):
    df = pd.read_csv(
        get_orig_path(year),
        sep="|",
        names=ORIG_COLS,
        header=None,
        dtype=str,
        low_memory=False
    )
    df["vintage_year"] = year
    return df

def load_svcg_year(year):
    df = pd.read_csv(
        get_svcg_path(year),
        sep="|",
        names=PERF_COLS,
        header=None,
        dtype=str,
        low_memory=False,
        index_col=False
    )
    df["vintage_year"] = year
    return df

## Note: memory-safe multi-year processing

The full multi-year servicing dataset is large enough to create memory pressure in Colab when loaded and typed all at once. To make the workflow more stable, each Freddie Mac vintage year will be processed separately, saved as a typed parquet file, and only then combined into unified multi-year origination and servicing datasets.

## 42. Create reusable typing functions


In [9]:
def type_orig_df(df):
    df = df.copy()

    for col in orig_date_cols:
        df[col] = pd.to_datetime(df[col], format="%Y%m", errors="coerce")

    for col in orig_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def type_svcg_df(df):
    df = df.copy()

    for col in svcg_date_cols:
        df[col] = pd.to_datetime(df[col], format="%Y%m", errors="coerce")

    for col in svcg_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

## 43. Process and save each vintage year separately

In [11]:
import pandas as pd

In [12]:
yearly_processing_summary = []

for year in YEARS:
    print(f"\nProcessing year {year}...")

    # Load raw year
    orig_year = load_orig_year(year)
    svcg_year = load_svcg_year(year)

    # Type year
    orig_year_typed = type_orig_df(orig_year)
    svcg_year_typed = type_svcg_df(svcg_year)

    # Save typed parquet files
    orig_out = PROCESSED / f"orig_{year}_typed.parquet"
    svcg_out = PROCESSED / f"svcg_{year}_typed.parquet"

    orig_year_typed.to_parquet(orig_out, index=False)
    svcg_year_typed.to_parquet(svcg_out, index=False)

    # Save lightweight summary
    yearly_processing_summary.append({
        "year": year,
        "orig_rows": orig_year_typed.shape[0],
        "orig_cols": orig_year_typed.shape[1],
        "svcg_rows": svcg_year_typed.shape[0],
        "svcg_cols": svcg_year_typed.shape[1],
        "orig_file": orig_out.name,
        "svcg_file": svcg_out.name
    })

    print(f"Saved {orig_out.name} and {svcg_out.name}")

    # Free memory aggressively
    del orig_year, svcg_year, orig_year_typed, svcg_year_typed


Processing year 2019...


/tmp/ipykernel_22218/255307672.py:20: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


Saved orig_2019_typed.parquet and svcg_2019_typed.parquet

Processing year 2020...


/tmp/ipykernel_22218/255307672.py:20: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


Saved orig_2020_typed.parquet and svcg_2020_typed.parquet

Processing year 2021...


...

Saved orig_2021_typed.parquet and svcg_2021_typed.parquet

Processing year 2022...


/tmp/ipykernel_22218/255307672.py:20: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


Saved orig_2022_typed.parquet and svcg_2022_typed.parquet

Processing year 2023...


/tmp/ipykernel_22218/255307672.py:20: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


Saved orig_2023_typed.parquet and svcg_2023_typed.parquet


## 44. Review yearly processing summary

In [13]:
yearly_processing_summary = pd.DataFrame(yearly_processing_summary)
yearly_processing_summary

,year,orig_rows,orig_cols,svcg_rows,svcg_cols,orig_file,svcg_file
0,2019,50000,33,1847477,32,orig_2019_typed.parquet,svcg_2019_typed.parquet
1,2020,50000,33,2340586,32,orig_2020_typed.parquet,svcg_2020_typed.parquet
2,2021,50000,33,2291290,32,orig_2021_typed.parquet,svcg_2021_typed.parquet
3,2022,50000,33,1788639,32,orig_2022_typed.parquet,svcg_2022_typed.parquet
4,2023,50000,33,1223737,32,orig_2023_typed.parquet,svcg_2023_typed.parquet


## 45. Save yearly processing summary

In [14]:
yearly_processing_summary.to_csv(
    TABLES / "yearly_processing_summary_2019_2023.csv",
    index=False
)

print(TABLES / "yearly_processing_summary_2019_2023.csv")

/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/yearly_processing_summary_2019_2023.csv


## 46. Combine saved typed origination files

In [15]:
orig_parquet_files = [
    PROCESSED / f"orig_{year}_typed.parquet"
    for year in YEARS
]

orig_all_typed = pd.concat(
    [pd.read_parquet(fp) for fp in orig_parquet_files],
    ignore_index=True
)

print("Combined origination shape:", orig_all_typed.shape)
display(orig_all_typed[[
    "loan_sequence_number",
    "vintage_year",
    "first_payment_date",
    "maturity_date",
    "credit_score",
    "cltv",
    "dti",
    "original_upb",
    "interest_rate"
]].head())

Combined origination shape: (250000, 33)


,loan_sequence_number,vintage_year,first_payment_date,maturity_date,credit_score,cltv,dti,original_upb,interest_rate
0,F19Q10000056,2019,2019-03-01,2049-02-01,741,80,33,372000,5.125
1,F19Q10000060,2019,2019-03-01,2049-02-01,731,77,44,250000,4.875
2,F19Q10000084,2019,2019-03-01,2049-02-01,722,95,41,261000,5.500
3,F19Q10000106,2019,2019-03-01,2049-02-01,729,87,17,61000,5.090
4,F19Q10000189,2019,2019-03-01,2049-02-01,773,29,43,390000,5.375


## 47. Combine saved typed servicing files

In [16]:
svcg_parquet_files = [
    PROCESSED / f"svcg_{year}_typed.parquet"
    for year in YEARS
]

svcg_all_typed = pd.concat(
    [pd.read_parquet(fp) for fp in svcg_parquet_files],
    ignore_index=True
)

print("Combined servicing shape:", svcg_all_typed.shape)
display(svcg_all_typed[[
    "loan_sequence_number",
    "vintage_year",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity",
    "current_interest_rate"
]].head())

Combined servicing shape: (9491729, 32)


,loan_sequence_number,vintage_year,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,current_interest_rate
0,F19Q10000056,2019,2019-02-01,372000.0,0,0,360,5.125
1,F19Q10000056,2019,2019-03-01,372000.0,0,1,359,5.125
2,F19Q10000056,2019,2019-04-01,371000.0,0,2,358,5.125
3,F19Q10000056,2019,2019-05-01,370000.0,0,3,357,5.125
4,F19Q10000056,2019,2019-06-01,370000.0,0,4,356,5.125


## 48. Build a multi-year combined dataset summary

In [17]:
multi_year_summary = pd.DataFrame({
    "dataset": ["orig_all_typed", "svcg_all_typed"],
    "rows": [orig_all_typed.shape[0], svcg_all_typed.shape[0]],
    "cols": [orig_all_typed.shape[1], svcg_all_typed.shape[1]]
})

multi_year_summary

,dataset,rows,cols
0,orig_all_typed,250000,33
1,svcg_all_typed,9491729,32


## 49. Validate vintage distributions

In [18]:
print("Origination vintage distribution:")
print(orig_all_typed["vintage_year"].value_counts().sort_index())

print("\nServicing vintage distribution:")
print(svcg_all_typed["vintage_year"].value_counts().sort_index())

Origination vintage distribution:
vintage_year
2019    50000
2020    50000
2021    50000
2022    50000
2023    50000
Name: count, dtype: int64

Servicing vintage distribution:
vintage_year
2019    1847477
2020    2340586
2021    2291290
2022    1788639
2023    1223737
Name: count, dtype: int64


## 50. Save combined multi-year datasets

In [19]:
orig_all_typed.to_parquet(PROCESSED / "orig_all_typed_2019_2023.parquet", index=False)
svcg_all_typed.to_parquet(PROCESSED / "svcg_all_typed_2019_2023.parquet", index=False)
multi_year_summary.to_csv(TABLES / "multi_year_ingestion_summary.csv", index=False)

print("Saved:")
print(PROCESSED / "orig_all_typed_2019_2023.parquet")
print(PROCESSED / "svcg_all_typed_2019_2023.parquet")
print(TABLES / "multi_year_ingestion_summary.csv")

Saved:
/content/drive/MyDrive/freddie_mac_risk_project/data/processed/orig_all_typed_2019_2023.parquet
/content/drive/MyDrive/freddie_mac_risk_project/data/processed/svcg_all_typed_2019_2023.parquet
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/multi_year_ingestion_summary.csv


## Conclusion

The validated Freddie Mac ingestion and typing workflow has now been scaled across all sample vintages from 2019 to 2023 using a memory-safe year-by-year process. Unified multi-year origination and servicing datasets have been created and saved, establishing the analytical base needed for 12-month serious delinquency target construction.

## Note: validate the servicing parser warning

The multi-year servicing data loaded successfully and the resulting fields look economically plausible, but pandas raised a parser warning indicating that the number of supplied column names may not fully match the raw row structure. Before moving into target creation, this step validates whether the warning is caused by a harmless trailing delimiter or by a meaningful schema mismatch.

## 52. Validate servicing row alignment against raw file

In [20]:
raw_svcg_path = INTERIM / "sample_2019" / "sample_svcg_2019.txt"

with open(raw_svcg_path, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()

raw_fields = first_line.split("|")

print("Raw field count from split('|'):", len(raw_fields))
print("\nFirst 12 raw fields:")
for i, val in enumerate(raw_fields[:12], start=1):
    print(f"{i}: {repr(val)}")

print("\nFirst parsed servicing row (key columns):")
display(
    svcg_all_typed.loc[svcg_all_typed["vintage_year"] == 2019, [
        "loan_sequence_number",
        "monthly_reporting_period",
        "current_actual_upb",
        "current_loan_delinquency_status",
        "loan_age",
        "remaining_months_to_legal_maturity",
        "defect_settlement_date",
        "modification_flag",
        "zero_balance_code",
        "zero_balance_effective_date",
        "current_interest_rate",
        "current_deferred_upb"
    ]].head(1)
)

Raw field count from split('|'): 32

First 12 raw fields:
1: 'F19Q10000056'
2: '201902'
3: '372000.00'
4: '0'
5: '000'
6: '360'
7: ''
8: ''
9: ''
10: ''
11: '5.125'
12: '0.00'

First parsed servicing row (key columns):


,loan_sequence_number,monthly_reporting_period,current_actual_upb,current_loan_delinquency_status,loan_age,remaining_months_to_legal_maturity,defect_settlement_date,modification_flag,zero_balance_code,zero_balance_effective_date,current_interest_rate,current_deferred_upb
0,F19Q10000056,2019-02-01,372000.0,0,0,360,None,None,None,NaT,5.125,0.0


## 53. Check for trailing empty field in raw servicing rows

In [21]:
print("Last 5 raw fields from first row:")
for i, val in enumerate(raw_fields[-5:], start=len(raw_fields)-4):
    print(f"{i}: {repr(val)}")

print("\nIs the final raw split field empty?", raw_fields[-1] == "")

Last 5 raw fields from first row:
28: ''
29: ''
30: ''
31: ''
32: '372000.00'

Is the final raw split field empty? False


## 54. Validate completeness of core servicing columns

In [22]:
core_svcg_cols = [
    "loan_sequence_number",
    "vintage_year",
    "monthly_reporting_period",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "remaining_months_to_legal_maturity"
]

core_svcg_quality = pd.DataFrame({
    "column": core_svcg_cols,
    "non_null_count": [svcg_all_typed[c].notna().sum() for c in core_svcg_cols],
    "total_rows": len(svcg_all_typed)
})

core_svcg_quality["non_null_rate"] = (
    core_svcg_quality["non_null_count"] / core_svcg_quality["total_rows"]
)

core_svcg_quality

,column,non_null_count,total_rows,non_null_rate
0,loan_sequence_number,9491729,9491729,1.0
1,vintage_year,9491729,9491729,1.0
2,monthly_reporting_period,9491729,9491729,1.0
3,current_actual_upb,9491729,9491729,1.0
4,current_loan_delinquency_status,9491729,9491729,1.0
5,loan_age,9491729,9491729,1.0
6,remaining_months_to_legal_maturity,9491729,9491729,1.0


## 55. Save servicing parser validation summary

In [23]:
core_svcg_quality.to_csv(TABLES / "core_servicing_quality_check.csv", index=False)

with open(TABLES / "raw_servicing_first_line_check.txt", "w") as f:
    f.write("Raw field count from split('|'): " + str(len(raw_fields)) + "\n")
    f.write("Final raw field empty: " + str(raw_fields[-1] == "") + "\n")
    f.write("First 12 raw fields:\n")
    for i, val in enumerate(raw_fields[:12], start=1):
        f.write(f"{i}: {repr(val)}\n")

print(TABLES / "core_servicing_quality_check.csv")
print(TABLES / "raw_servicing_first_line_check.txt")

/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/core_servicing_quality_check.csv
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/raw_servicing_first_line_check.txt


## Conclusion

The servicing parser warning has now been investigated directly by comparing raw servicing rows with parsed dataframe values and by checking whether the apparent extra field is a trailing empty value. If the key business fields align and the core servicing columns are complete, the multi-year servicing dataset can be treated as safe for 12-month serious delinquency target construction.